# Processing SLF Observation Data for Machine Learning Models
## Erica Keklak | 2025-06-25 to present
Estimating how many spotted lanternflies are in a county and state per year is a vital part of two features in each model's training dataset: the previous year's abundance and the current year's abundance. We have to estimate data using a multiplier from how many observations were made in a county per year, the average number of lanternflies observed per observation, and how many lanternflies can occupy that county given its natural resources.

In [70]:
# Import necessary libraries and packages

import numpy as np
import pandas as pd
import geopandas as gpd
import datetime
import shapefile
from shapely.geometry import shape, Point
import matplotlib
import matplotlib.cm as cm
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

In [3]:
# Read in files from geometry data processing

county_path = 'C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Modified Data Backups/Geometry/county_geometry.shp'
county_gdf = gpd.read_file(county_path)
county_shp = shapefile.Reader(county_path)
county_shapes = county_shp.shapes()

state_path = 'C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Modified Data Backups/Geometry/state_geometry.shp'
state_gdf = gpd.read_file(state_path)
state_shp = shapefile.Reader(state_path)
state_shapes = state_shp.shapes()

county_annual_gdf = gpd.read_file('C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Modified Data Backups/Geometry/county_geometry_annual.shp')
state_annual_gdf = gpd.read_file('C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Modified Data Backups/Geometry/state_geometry_annual.shp')

In [20]:
# Read iNaturalist spotted lanternfly observations

slf_iNat = pd.read_csv('C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Raw Data Backups/Lycorma delicatula Ecological/iNaturalist/Lycorma delicatula NEW.csv/Lycorma delicatula NEW.csv',
                       dtype = {31: str, 42: str}
                      )

# Read IECO Lab spotted lanternfly observations

# slf_ieco_gdf = gpd.read_file('C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Raw Data Backups/Lycorma delicatula Ecological/EDDMapS Bugwood SLF Reports/observations.gpkg',
#                              layer = 'PolygonLayer'
#                             )

# Read Lydemapr observations
# Citation: De Bona, S., L. Barringer, P. Kurtz, J. Losiewicz, G.R. Parra, & M.R. Helmus. lydemapr: an R package to track the spread of the invasive 
#           spotted lanternfly (Lycorma delicatula, White 1845) (Hemiptera, Fulgoridae) in the United States. NeoBiota 86: 151-168.
#           https://doi.org/10.3897/neobiota.86.101471

slf_lydemapr = pd.read_csv('C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Raw Data Backups/Lycorma delicatula Ecological/iEco Lab/lyde_data_v3.0.3/lyde_data_v3.0.3/lyde.csv',
                           dtype = {7: str, 8: str}
                          )

# Convert to DataFrames if in dictionary form

slf_iNat_df = pd.DataFrame(slf_iNat)
slf_lyde_df = pd.DataFrame(slf_lydemapr)

## Part 1/6: Process iNaturalist Data Basics

In [22]:
# Observe the columns and contents of the iNaturalist DataFrame

slf_iNat_df.head()

,id,uuid,observed_on_string,observed_on,time_observed_at,time_zone,user_id,user_login,user_name,created_at,...,positioning_device,place_county_name,place_state_name,species_guess,scientific_name,common_name,iconic_taxon_name,taxon_id,field:quantity,field:quantity observed
0,3842901,4de0ab2b-e119-4d5d-9708-f089926868f0,2016-08-06 11:44:00 AM EDT,2016-08-06,2016-08-06 15:44:00 +0000,Eastern Time (US & Canada),298484,suzyb,NaN,2016-08-09 14:52:13 +0000,...,gps,Berks,Pennsylvania,Spotted Lanternfly,Lycorma delicatula,Spotted Lanternfly,Insecta,324726,NaN,NaN
1,4066989,8b45aa6e-74f4-47d8-9961-2cda8bd77f68,9/8/2016 5:52:24 PM,2016-09-08,2016-09-08 21:52:24 +0000,Eastern Time (US & Canada),309360,jasondalesmith,Jason Langkamer-Smith,2016-09-08 21:52:25 +0000,...,NaN,Lehigh,Pennsylvania,Spotted Lanternfly,Lycorma delicatula,Spotted Lanternfly,Insecta,324726,NaN,NaN
2,4286849,eef017e3-bf5a-4e3f-b906-6ec0ae1249fe,2016-10-06 5:43:11 PM EDT,2016-10-06,2016-10-06 21:43:11 +0000,Eastern Time (US & Canada),342673,nmey13,NaN,2016-10-06 21:48:45 +0000,...,gps,Berks,Pennsylvania,Spotted Lanternfly,Lycorma delicatula,Spotted Lanternfly,Insecta,324726,NaN,NaN
3,5512985,3fc0c98d-3f00-46f7-a497-3733fba7666c,Sun Oct 02 2016 11:16:58 GMT-0400 (EDT),2016-10-02,2016-10-02 15:16:58 +0000,Eastern Time (US & Canada),437734,futurescientist,NaN,2017-03-31 00:30:54 +0000,...,NaN,Berks,Pennsylvania,Spotted Lanternfly,Lycorma delicatula,Spotted Lanternfly,Insecta,324726,NaN,NaN
4,6564552,22821c43-4904-406d-a1f9-764ce4b96a5a,Fri Jun 09 2017 12:53:16 GMT-0400 (EDT),2017-06-09,2017-06-09 16:53:16 +0000,Eastern Time (US & Canada),467834,evantomology,Evan Waite,2017-06-09 20:18:11 +0000,...,NaN,Berks,Pennsylvania,Spotted Lanternfly,Lycorma delicatula,Spotted Lanternfly,Insecta,324726,NaN,NaN


In [24]:
# Delete columns automatically included in the query that were added to the iNat DataFrame but have no use in this project

slf_iNat_df = slf_iNat.drop(columns = ['id', 'uuid', 'observed_on_string', 'user_id', 'user_login', 'user_name', 'updated_at', 'license', 'url',
                                       'image_url', 'sound_url', 'num_identification_agreements', 'num_identification_disagreements',
                                       'captive_cultivated', 'oauth_application_id', 'description', 'place_guess', 'private_place_guess', 'geoprivacy',
                                       'taxon_geoprivacy', 'coordinates_obscured', 'positioning_method', 'positioning_device', 'species_guess',
                                       'scientific_name', 'common_name', 'iconic_taxon_name', 'taxon_id', 'field:quantity'
                                      ], axis = 1)


slf_iNat_df.columns

Index(['observed_on', 'time_observed_at', 'time_zone', 'created_at',
       'quality_grade', 'tag_list', 'latitude', 'longitude',
       'positional_accuracy', 'private_latitude', 'private_longitude',
       'public_positional_accuracy', 'place_county_name', 'place_state_name',
       'field:quantity observed'],
      dtype='object')

In [26]:
# Find dates that don't appear within 2014-01-01 to 2024-12-31 UTC

too_early = []
too_late = []

for i in range(0, len(slf_iNat_df)):
    if int(slf_iNat_df['observed_on'][i][0:4]) < 2014:
        too_early = too_early.append([i])
    elif int(slf_iNat_df['observed_on'][i][0:4]) > 2024:
        too_late = too_late.append([i])

too_early
too_late

[]

In [27]:
# Find more dates that don't appear within 2014-01-01 to 2024-12-31 UTC just to be sure

pd.unique(slf_iNat_df['time_observed_at'])

array(['2016-08-06 15:44:00 +0000', '2016-09-08 21:52:24 +0000',
       '2016-10-06 21:43:11 +0000', ..., '2024-07-21 13:39:57 +0000',
       '2021-06-26 15:04:58 +0000', '2024-09-06 20:31:16 +0000'],
      dtype=object)

In [30]:
# Find time zones that don't fit within the 48 United States of concern

pd.unique(slf_iNat_df['time_zone'])

array(['Eastern Time (US & Canada)', 'Central Time (US & Canada)',
       'Auckland', 'UTC', 'Baghdad', 'Hawaii',
       'Mountain Time (US & Canada)', 'Atlantic Time (Canada)',
       'Pacific Time (US & Canada)', 'Indiana (East)', 'America/New_York',
       'Africa/Johannesburg', 'Bangkok', 'Samoa', 'Edinburgh',
       'America/Detroit', 'Amsterdam', 'Lima', 'Arizona', 'Santiago',
       'Puerto Rico', 'Athens', 'Bogota', 'America/Indiana/Vevay'],
      dtype=object)

In [32]:
# Delete data with time zones that aren't needed

slf_iNat_df = slf_iNat_df.drop(slf_iNat_df[slf_iNat_df['time_zone'] == 'Auckland'].index)
slf_iNat_df = slf_iNat_df.drop(slf_iNat_df[slf_iNat_df['time_zone'] == 'UTC'].index)
slf_iNat_df = slf_iNat_df.drop(slf_iNat_df[slf_iNat_df['time_zone'] == 'Baghdad'].index)
slf_iNat_df = slf_iNat_df.drop(slf_iNat_df[slf_iNat_df['time_zone'] == 'Hawaii'].index)
slf_iNat_df = slf_iNat_df.drop(slf_iNat_df[slf_iNat_df['time_zone'] == 'Africa/Johannesburg'].index)
slf_iNat_df = slf_iNat_df.drop(slf_iNat_df[slf_iNat_df['time_zone'] == 'Bangkok'].index)
slf_iNat_df = slf_iNat_df.drop(slf_iNat_df[slf_iNat_df['time_zone'] == 'Samoa'].index)
slf_iNat_df = slf_iNat_df.drop(slf_iNat_df[slf_iNat_df['time_zone'] == 'Edinburgh'].index)
slf_iNat_df = slf_iNat_df.drop(slf_iNat_df[slf_iNat_df['time_zone'] == 'Amsterdam'].index)
slf_iNat_df = slf_iNat_df.drop(slf_iNat_df[slf_iNat_df['time_zone'] == 'Lima'].index)
slf_iNat_df = slf_iNat_df.drop(slf_iNat_df[slf_iNat_df['time_zone'] == 'Puerto Rico'].index)
slf_iNat_df = slf_iNat_df.drop(slf_iNat_df[slf_iNat_df['time_zone'] == 'Athens'].index)
slf_iNat_df = slf_iNat_df.drop(slf_iNat_df[slf_iNat_df['time_zone'] == 'Bogota'].index)

# Find more dates that don't appear within 2014-01-01 to 2024-12-31 UTC

pd.unique(slf_iNat_df['created_at'])

array(['2016-08-09 14:52:13 +0000', '2016-09-08 21:52:25 +0000',
       '2016-10-06 21:48:45 +0000', ..., '2025-07-04 11:59:01 +0000',
       '2025-07-04 17:00:59 +0000', '2025-07-04 18:15:21 +0000'],
      dtype=object)

In [34]:
# Delete data with latitude coordinates not found in the lower 48 United States

slf_iNat_df = slf_iNat_df.drop(slf_iNat_df[slf_iNat_df['latitude'] > 50].index)
slf_iNat_df = slf_iNat_df.drop(slf_iNat_df[slf_iNat_df['latitude'] < 23].index)

# Delete data with longitude coordinates not found in the lower 48 United States

slf_iNat_df = slf_iNat_df.drop(slf_iNat_df[slf_iNat_df['longitude'] > -66].index)
slf_iNat_df = slf_iNat_df.drop(slf_iNat_df[slf_iNat_df['longitude'] < -126].index)

In [36]:
# Find observations with positional accuracies greater than an unusual amount (these are measured in meters)

pd.unique(slf_iNat_df['positional_accuracy'])

array([  344.,    nan, 11007., ...,  4333.,  3384., 13144.])

In [38]:
# Delete data with unusually large locational precision values (low precision)

slf_iNat_df = slf_iNat_df.drop(slf_iNat_df[slf_iNat_df['positional_accuracy'] > 5000].index)

# Find observations with publicized positional accuracies greater than an unusual amount (these are measured in meters)

pd.unique(slf_iNat_df['public_positional_accuracy'])

array([ 344.,   nan,  246., ..., 4514., 4333., 3384.])

In [40]:
# Delete data with unusually large private locational precicion values (low precision)

slf_iNat_df = slf_iNat_df.drop(slf_iNat_df[slf_iNat_df['public_positional_accuracy'] > 5000].index)

# Find observations with state names that do not make sense

pd.unique(slf_iNat_df['place_state_name'])

array(['Pennsylvania', 'New Jersey', 'Virginia', 'Delaware', 'New York',
       'Maryland', 'Massachusetts', 'Utah', 'Michigan', 'West Virginia',
       'North Carolina', 'Connecticut', 'Vermont', 'Florida', 'Tennessee',
       'Ohio', 'New Hampshire', 'Indiana', 'Rhode Island',
       'District of Columbia', 'Illinois', 'California', 'Wisconsin',
       'Kentucky', 'Oregon'], dtype=object)

In [42]:
# Put private coordinates into public coordinates if they are missing

for i in range(0, len(slf_iNat_df)):
    latitude = slf_iNat_df.iloc[i].loc['latitude']
    longitude = slf_iNat_df.iloc[i].loc['longitude']
    p_latitude = slf_iNat_df.iloc[i].loc['private_latitude']
    p_longitude = slf_iNat_df.iloc[i].loc['private_longitude']
    if (type(latitude) != np.float64 or latitude < 23 or latitude > 50) and (type(p_latitude) == np.float64 and p_latitude >= 23 and p_latitude <= 50):
        slf_iNat_df.at[i, 'latitude'] = slf_iNat_df.at[i, 'private_latitude']
    elif (type(latitude) != np.float64 or latitude < 23 or latitude > 50) and type(p_latitude) != np.float64:
        print('Cannot add latitude because private latitude isn\'t available')
    if (type(longitude) != np.float64 or longitude < -126 or longitude > -66) and (type(p_longitude) == np.float64 and p_longitude >= -126 and p_longitude <= -66):
        slf_iNat_df.at[i, 'longitude'] = slf_iNat_df.at[i, 'private_longitude']
    elif (type(longitude) != np.float64 or longitude < -126 or longitude > -66) and type(p_longitude) != np.float64:
        print('Cannot add longitude because private longitude isn\'t available')

In [43]:
# Now clean up the data to only include the most important values

slf_iNat_df = slf_iNat_df.drop(columns = ['observed_on', 'time_zone', 'created_at', 'quality_grade', 'tag_list', 'positional_accuracy', 'private_latitude',
                                          'private_longitude', 'public_positional_accuracy'
                                         ], axis = 1)

# Convert the observation time into datetimes and delete the old column

def to_time_iNat(column):
    list_times = []
    for timestamp in range(0, len(column)):
        new_timestamp = str(column.iloc[timestamp])
        new_timestamp = datetime.datetime.strptime(new_timestamp, '%Y-%m-%d %H:%M:%S +0000')
        list_times.append(new_timestamp)
    return list_times

slf_iNat_df = slf_iNat_df.dropna(subset = ['time_observed_at'])
slf_iNat_df['timestamp'] = to_time_iNat(slf_iNat_df['time_observed_at'])
slf_iNat_df['year'] = [i.year for i in slf_iNat_df['timestamp']]
slf_iNat_df['year']

# Sum each county's total observations

slf_iNat_df.groupby(['place_county_name','place_state_name', 'year']).count()

time_observed_at  latitude  \
place_county_name place_state_name year                               
Adams             Pennsylvania     2021                 1         1   
                                   2022                12        12   
                                   2023                29        29   
                                   2024                73        73   
Alameda           California       2023                 1         1   
...                                                   ...       ...   
York              Pennsylvania     2021                69        69   
                                   2022               117       117   
                                   2023                27        27   
                                   2024                31        31   
                  Virginia         2022                 1         1   

                                         longitude  field:quantity observed  \
place_county_name place_state_name year                                       
Adams             Pennsylvania     2021          1                        0   
                                   2022         12                        0   
                                   2023         29                        0   
                                   2024         73                        0   
Alameda           California       2023          1                        0   
...                                            ...                      ...   
York              Pennsylvania     2021         69                        0   
                                   2022        117                        0   
                                   2023         27                        0   
                                   2024         31                        0   
                  Virginia         2022          1                        0   

                                         timestamp  
place_county_name place_state_name year             
Adams             Pennsylvania     2021          1  
                                   2022         12  
                                   2023         29  
                                   2024         73  
Alameda           California       2023          1  
...                                            ...  
York              Pennsylvania     2021         69  
                                   2022        117  
                                   2023         27  
                                   2024         31  
                  Virginia         2022          1  

[786 rows x 5 columns]

In [44]:
# Make a column for each observation's season

# NYMPHS: April to September
# ADULTS: July to November
# EGGS: September to June
# january: egg
# february: egg
# march: egg
# april: egg nymph
# may: egg nymph
# june: egg nymph
# july nymph adult
# august: nymph adult
# september: egg nymph adult
# october: adult egg
# november: adult egg
# december: egg

def find_season(column):
    list_season_ids = []
    for i in range(0, len(column)):
        month = column.iloc[i].month
        if month < 4 or month == 12:
            list_season_ids = list_season_ids + ['egg']
        elif month < 7:
            list_season_ids = list_season_ids + ['egg nymph']
        elif month < 9:
            list_season_ids = list_season_ids + ['nymph adult']
        elif month < 12:
            list_season_ids = list_season_ids + ['adult egg']
        else:
            print('This (',column[i].month,') doesn\'t represent a month.')
    return list_season_ids

slf_iNat_df['season'] = find_season(slf_iNat_df['timestamp'])

## Part 2/6: Process iNaturalist "Quantity of Spotted Lanternflies Observed" Data by County

In [46]:
# Sum each county's total annual individual SLFs observed

observed_quantities_county = {}
observations_made_county = {}
county_years = []
latitudes_county = {}
longitudes_county = {}

x = pd.DataFrame({})
x['county'] = slf_iNat_df['place_county_name']
x['state'] = slf_iNat_df['place_state_name']
x['year'] = slf_iNat_df['year']
x['lat'] = slf_iNat_df['latitude']
x['long'] = slf_iNat_df['longitude']
x['qty'] = slf_iNat_df['field:quantity observed']
x = x.reset_index(drop = True)
x['qty'] = x['qty'].fillna(1)
x.head()

,county,state,year,lat,long,qty
0,Berks,Pennsylvania,2016,40.380610,-75.719145,1.0
1,Lehigh,Pennsylvania,2016,40.592449,-75.521499,1.0
2,Berks,Pennsylvania,2016,40.476929,-75.624994,1.0
3,Berks,Pennsylvania,2017,40.411820,-75.660033,1.0
4,Berks,Pennsylvania,2017,40.310517,-75.725005,1.0


In [47]:
z = x.loc[x['county'] == 'Ocean']
z = z.loc[z['state'] == 'New Jersey']
z = z.loc[z['year'] == 2024]
len(z)

21

In [48]:
for i in range(0, len(x)):
    county = x.iloc[i].loc['county']
    state = x.iloc[i].loc['state']
    year = x.iloc[i].loc['year']
    county_year = str(county + ' County, ' + state + ', ' + str(year))
    
    if county_year not in county_years:
        county_years = county_years + [county_year]
        observed_quantities_county[county_year] = x.at[i, 'qty']
        observations_made_county[county_year] = 1
        latitudes_county[county_year] = [x.at[i, 'lat'].item()]
        longitudes_county[county_year] = [x.at[i, 'long'].item()]
        
    else:
        observed_quantities_county[county_year] = observed_quantities_county[county_year] + x.at[i, 'qty']
        observations_made_county[county_year] = observations_made_county[county_year] + 1
        if county_year in latitudes_county:
            latitudes_county[county_year] = list(latitudes_county[county_year] + [x.at[i, 'lat']])
        else:
            latitudes_county[county_year] = [x.at[i, 'lat'].item()]
        if county_year in longitudes_county:
            longitudes_county[county_year] = list(longitudes_county[county_year] + [x.at[i, 'long']])
        else:
            longitudes_county[county_year] = [x.at[i, 'long'].item()]

In [49]:
# Add these quantities to the annual county GeoDataFrame with data points by default containing a quantity 0

county_annual_gdf['slfs_obser'] = [0] * len(county_annual_gdf)
county_annual_gdf['slf_observ'] = [0] * len(county_annual_gdf)
county_annual_gdf['lats'] = [0] * len(county_annual_gdf)
county_annual_gdf['longs'] = [0] * len(county_annual_gdf)
county_annual_gdf['lats'] = county_annual_gdf['lats'].astype('object')
county_annual_gdf['longs'] = county_annual_gdf['longs'].astype('object')

In [50]:
for i in range(0, len(county_years)):
    point = county_years[i]
    desired_county = str(point[0 : point.find(' County')])
    desired_state = str(point[(point.find(', ') + 2) : (len(point) - 6)])
    desired_year = int(point[(len(point) - 5) : len(point)])
    for j in range(0, len(county_annual_gdf)):
        if desired_county == str(county_annual_gdf.iloc[j].loc['county']) and desired_state == str(county_annual_gdf.iloc[j].loc['state']) and desired_year == int(county_annual_gdf.iloc[j].loc['year']):
            county_annual_gdf.at[j, 'slfs_obser'] = observed_quantities_county[point]
            county_annual_gdf.at[j, 'slf_observ'] = observations_made_county[point]
            county_annual_gdf.at[j, 'lats'] = latitudes_county[point]
            county_annual_gdf.at[j, 'longs'] = longitudes_county[point]
            print('Data point added')

Data point added
Data point added
Data point added
Data point added
Data point added
Data point added
Data point added
Data point added
Data point added
Data point added
Data point added
Data point added
Data point added
Data point added
Data point added
Data point added
Data point added
Data point added
Data point added
Data point added
Data point added
Data point added
Data point added
Data point added
Data point added
Data point added
Data point added
Data point added
Data point added
Data point added
Data point added
Data point added
Data point added
Data point added
Data point added
Data point added
Data point added
Data point added
Data point added
Data point added
Data point added
Data point added
Data point added
Data point added
Data point added
Data point added
Data point added
Data point added
Data point added
Data point added
Data point added
Data point added
Data point added
Data point added
Data point added
Data point added
Data point added
Data point added
Data point add

KeyboardInterrupt: 

In [ ]:
nj_popped = county_annual_gdf.loc[county_annual_gdf['slfs_obser'] != 0]
nj_popped = nj_popped.loc[nj_popped['state'] == 'New Jersey']
pd.unique(nj_popped['year'])

In [ ]:
# Get yesteryear's quantities observed and numbers of observations and apply to county data

observed_last_year = []
observations_last_year = []

for i in range(0, len(county_annual_gdf)):
    if i < 3108:
        observed_last_year = observed_last_year + [0]
        observations_last_year = observations_last_year + [0]
    else:
        observed_last_year = observed_last_year + [county_annual_gdf.at[i - 3108, 'slfs_obser']]
        observations_last_year = observations_last_year + [county_annual_gdf.at[i - 3108, 'slf_observ']]

county_annual_gdf['y_slfs_obs'] = observed_last_year
county_annual_gdf['y_slf_obse'] = observations_last_year

In [ ]:
county_annual_gdf.loc[county_annual_gdf['y_slfs_obs'] != 0].head()

In [ ]:
# Alternative excluding latitudes and longitudes

# county_annual_gdf['slfs_obser'] = [0] * len(county_annual_gdf)
# county_annual_gdf['slf_observ'] = [0] * len(county_annual_gdf)

# for i in range(0, len(county_years)):
#     desired_county = str(county_years[i][0 : county_years[i].find(' County')])
#     desired_state = str(county_years[i][(county_years[i].find(', ') + 2) : (len(county_years[i]) - 6)])
#     desired_year = int(county_years[i][(len(county_years[i]) - 5) : len(county_years[i])])
#     for j in range(0, len(county_annual_gdf)):
#         county_now = str(county_annual_gdf.iloc[j].loc['county'])
#         state_now = str(county_annual_gdf.iloc[j].loc['state'])
#         year_now = int(county_annual_gdf.iloc[j].loc['year'])
#         if desired_county == county_now and desired_state == state_now and desired_year == year_now:
#             county_annual_gdf.at[j, 'slfs_obser'] = observed_quantities_county[county_years[i]]
#             county_annual_gdf.at[j, 'slf_observ'] = observations_made_county[county_years[i]]

In [ ]:
# Store these data because it took a long time to process

county_annual_gdf.to_file('C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Modified Data Backups/county_annual_with_slf.shp')

In [ ]:
# Storing as an shp doesn't always work so I'm going to convert it into a csv as well

county_annual_gdf.to_csv('C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Modified Data Backups/county_annual_with_slf.csv')

In [72]:
county_annual_data = gpd.read_file('C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Modified Data Backups/county_annual_with_slf.shp')
county_annual_data.head()

,county,state,land_area,water_area,year,slfs_obser,slf_observ,lats,longs,y_slfs_obs,y_slf_obse,geometry
0,Houston,Alabama,1501742250,4795418,2014,0,0,0,0,0,0,"POLYGON ((-85.712 31.197, -85.709 31.198, -85...."
1,Choctaw,Alabama,2365900084,19114321,2014,0,0,0,0,0,0,"POLYGON ((-88.473 31.894, -88.469 31.93, -88.4..."
2,Russell,Alabama,1660653961,15562947,2014,0,0,0,0,0,0,"POLYGON ((-85.435 32.318, -85.434 32.392, -85...."
3,Sussex,Delaware,2424590442,674129051,2014,0,0,0,0,0,0,"POLYGON ((-75.723 38.83, -75.615 38.834, -75.5..."
4,Jackson,Alabama,2792044612,126334711,2014,0,0,0,0,0,0,"MULTIPOLYGON (((-86.154 34.534, -86.15 34.534,..."


## Part 3/6: Process iNaturalist "Quantity of Spotted Lanternflies Observed" Data by State

In [ ]:
# Make a new DataFrame containing only the iNaturalist dataset's important columns for state data processing

y = pd.DataFrame({})
y['state'] = slf_iNat_df['place_state_name']
y['year'] = slf_iNat_df['year']
y['qty'] = slf_iNat_df['field:quantity observed']
y['lat'] = slf_iNat_df['latitude']
y['long'] = slf_iNat_df['longitude']
y = y.reset_index(drop = True)
y['qty'] = y['qty'].fillna(1)
y.head()

In [ ]:
# Process the new DataFrame by each row

observed_quantities_state = {}
state_years = []
observations_made_state = {}
latitudes_state = {}
longitudes_state = {}

for i in range(0, len(y)):
    state = y.iloc[i].loc['state']
    year = y.iloc[i].loc['year']
    state_year = str(state + ', ' + str(year))
    
    if state_year not in state_years: # Assesses if there aren't any state-and-year-specific data points currently in the state_years list
        state_years = state_years + [state_year] # Adds the new name to the list
        observed_quantities_state[state_year] = y.at[i, 'qty'] # Assigns the first observation's quantity to the dictionary
        observations_made_state[state_year] = 1
        latitudes_state[state_year] = [y.at[i, 'lat'].item()]
        longitudes_state[state_year] = [y.at[i, 'long'].item()]
        
    else:
        observed_quantities_state[state_year] = observed_quantities_state[state_year] + y.at[i, 'qty'] # Updates the observation's state's quantity
        observations_made_state[state_year] = observations_made_state[state_year] + 1
        if state_year in latitudes_state:
            latitudes_state[state_year] = list(latitudes_state[state_year] + [y.at[i, 'lat']])
        else:
            latitudes_state[state_year] = [y.at[i, 'lat'].item()]
        if state_year in longitudes_state:
            longitudes_state[state_year] = list(longitudes_state[state_year] + [y.at[i, 'long']])
        else:
            longitudes_state[state_year] = [y.at[i, 'long'].item()]

In [ ]:
state_annual_gdf['slfs_obser'] = [0] * len(state_annual_gdf)
state_annual_gdf['slf_observ'] = [0] * len(state_annual_gdf)

for i in range(0, len(state_years)):
    desired_state = str(state_years[i][0 : (len(state_years[i]) - 6)])
    desired_year = int(state_years[i][(len(state_years[i]) - 4) : len(state_years[i])])
    for j in range(0, len(state_annual_gdf)):
        if desired_state == str(state_annual_gdf.iloc[j].loc['state']) and desired_year == int(state_annual_gdf.iloc[j].loc['year']):
            state_annual_gdf.at[j, 'slfs_obser'] = observed_quantities_state[state_years[i]]
            state_annual_gdf.at[j, 'slf_observ'] = observations_made_state[state_years[i]]

state_annual_gdf.head()

In [ ]:
# Get yesteryear's quantities observed and numbers of observations and apply to state data

observed_last_year = []
observations_last_year = []

for i in range(0, len(state_annual_gdf)):
    if i < 48:
        observed_last_year = observed_last_year + [0]
        observations_last_year = observations_last_year + [0]
    else:
        observed_last_year = observed_last_year + [state_annual_gdf.at[i - 48, 'slfs_obser']]
        observations_last_year = observations_last_year + [state_annual_gdf.at[i - 48, 'slf_observ']]

state_annual_gdf['y_slfs_obs'] = observed_last_year
state_annual_gdf['y_slf_obse'] = observations_last_year

In [ ]:
state_annual_gdf.loc[state_annual_gdf['slfs_obser'] != 0]

In [ ]:
# Store these data because it took a long time to process

state_annual_gdf.to_file('C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Modified Data Backups/state_annual_with_slf.shp')

In [ ]:
# Storing as an shp doesn't always work so I'm going to convert it into a csv as well

state_annual_gdf.to_csv('C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Modified Data Backups/state_annual_with_slf.csv')

In [74]:
state_annual_data = gpd.read_file('C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Modified Data Backups/state_annual_with_slf.shp')
state_annual_data.head()

,state,land_area,water_area,year,slfs_obser,slf_observ,y_slfs_obs,y_slf_obse,geometry
0,New Mexico,314198519809,726531289,2014,0,0,0,0,"POLYGON ((-109.05 31.48, -109.05 31.5, -109.05..."
1,South Dakota,196341670967,3387563375,2014,0,0,0,0,"POLYGON ((-104.06 44.998, -104.05 44.998, -104..."
2,California,403673433805,20291632828,2014,0,0,0,0,"MULTIPOLYGON (((-118.6 33.479, -118.6 33.478, ..."
3,Kentucky,102266755818,2384136185,2014,0,0,0,0,"MULTIPOLYGON (((-89.406 36.528, -89.399 36.542..."
4,Alabama,131185561946,4581813708,2014,0,0,0,0,"MULTIPOLYGON (((-88.053 30.507, -88.051 30.509..."


## Part 4/6: Process IECO Lab/Lydemapr Data

In [ ]:
# Observe the columns and contents of the IECO Lab DataFrame

slf_lyde_df.head()

In [ ]:
# Remove unnecessary columns; we will be using the "bio year" instead of the year of observation to keep the next year's generation of eggs out of the
# current year's population

slf_lyde_df = slf_lyde_df.drop(columns = ['survey', 'source_agency', 'collection_method', 'pointID', 'rounded_longitude_10k', 'rounded_latitude_10k'],
                               axis = 1)
slf_lyde_df.head()

In [ ]:
# Remove data from non-contiguous U.S. coordinates

slf_lyde_df = slf_lyde_df.drop(slf_lyde_df[slf_lyde_df['latitude'] < 23].index, axis = 0)
slf_lyde_df = slf_lyde_df.drop(slf_lyde_df[slf_lyde_df['latitude'] > 50].index, axis = 0)
slf_lyde_df = slf_lyde_df.drop(slf_lyde_df[slf_lyde_df['longitude'] < -126].index, axis = 0)
slf_lyde_df = slf_lyde_df.drop(slf_lyde_df[slf_lyde_df['longitude'] > -66].index, axis = 0)

slf_lyde_df.head()

In [ ]:
pd.unique(slf_lyde_df['state'])

# All of these are contiguous states

In [ ]:
pd.unique(slf_lyde_df['lyde_density'])

In [ ]:
# Replace unknown densities with Unpopulated

slf_lyde_df['lyde_density'] = slf_lyde_df['lyde_density'].fillna('Unpopulated')

In [ ]:
pd.unique(slf_lyde_df['lyde_density'])

In [290]:
# Assign a county column to each data point based on coordinates

county_guess = []

for i in range(0, len(slf_lyde_df)):
    lat = slf_lyde_df.at[i, 'latitude']
    long = slf_lyde_df.at[i, 'longitude']
    point = Point(long, lat)
    county_not_found = True
    for j in range(0, len(county_gdf)): # look through county geometry to see if there is anything to be done
        polygon = shape(county_shapes[j]) # might not work for multipolygons
        if polygon.contains(point):
            county_guess = county_guess + [county_gdf.at[j, 'county']]
            county_not_found = False
    if county_not_found:
        county_guess = county_guess + ['Unknown']
        
    # Find all possible counties for the state provided using county_gdf given the state cell in the row
    # Pass this list of counties and geometries, then 
    # Iterate through the counties to try to find if the observation fits
    # Iterate through the polygons to try to find if an observation fits
    
slf_lyde_df['county'] = county_guess

KeyboardInterrupt: 

In [ ]:
# Save modified Lydemap data; can't save it as a shapefile because it doesn't have geometry

slf_lyde_df.to_csv('C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Modified Data Backups/lydemap_with_county.csv')

In [ ]:
# pts = shapefile.Reader("points.shp")
# polys = shapefile.Reader("multipol.shp")
# points = [pt.shape.__geo_interface__ for pt in pts.shapeRecords()]
# multi = shape(polys.shapeRecords()[0].shape.__geo_interface__) # 1 polygon
# print multi
# >> MULTIPOLYGON (((-0.5275288092189501 0.5569782330345711, -0.117797695262484 0.2906530089628682, -0.2560819462227913 0.01920614596670933, -0.7093469910371319 -0.08834827144686286, -0.8629961587708066 0.1830985915492958, -0.734955185659411 0.3982074263764405, -0.5275288092189501 0.5569782330345711)), ((0.1997439180537772 0.06017925736235596, 0.5480153649167734 0.1293213828425096, 0.729833546734955 0.03969270166453265, 0.8143405889884763 -0.1395646606914211, 0.701664532650448 -0.3854033290653009, 0.4763124199743918 -0.5006402048655569, 0.2688860435339309 -0.4238156209987196, 0.1895006402048656 -0.2291933418693981, 0.1997439180537772 0.06017925736235596)), ((-0.3764404609475033 -0.295774647887324, -0.1152368758002562 -0.3597951344430217, -0.03329065300896294 -0.5800256081946222, -0.1152368758002562 -0.7413572343149808, -0.3072983354673495 -0.8591549295774648, -0.58898847631242 -0.6927016645326505, -0.6555697823303457 -0.4750320102432779, -0.3764404609475033 -0.295774647887324)))
# for i, pt in enumerate(points):
#     point = shape(pt)
#     if point.within(multi): 
#         print i, shape(points[i])

In [ ]:
# read your shapefile
# r = shapefile.Reader("your_shapefile.shp")

# get the shapes
# shapes = r.shapes() # county_shapes and state_shapes

# build a shapely polygon from your shape
# polygon = shape(shapes[0])    

# def check(lon, lat):
    # build a shapely point from your geopoint
#     point = Point(lon, lat)

    # the contains function does exactly what you want
#     return polygon.contains(point)

## Part 5/6: Remove Duplicate Data Between Both Datasets

In [ ]:
# Find iNat coordinates and observation years the same as the lydemap iNat-sourced ones (use year, not bio_year, but pass the bio_year to the final dataset
# because that's how we make the best estimates

lyde_from_iNat = slf_lyde_df.loc[slf_lyde_df['source'] == 'inat']
lyde_from_other = slf_lyde_df.loc[slf_lyde_df['source'] != 'inat']

In [ ]:
shared_observations = []

for i in range(0, len(slf_iNat_df)): # choosing this one to iterate on since there are more slf_iNat_df data points than lyde_from_iNat
    for j in range(0, len(lyde_from_iNat)):
        if slf_iNat_df.at[i, ]

In [ ]:
# Check each iNat and Lydemapr observation and make sure that the aggregate is a union, not an intersection, between the observations made in both

unique_observations = []
for i in range(0, max(len(slf_iNat_df), len(slf_lyde_df))):
    lydemap_observation = slf_lyde_df.iloc[i]
    lyde_year = slf_lyde_df.iloc[i].loc['year']
    if pass: # insert qualifying factors here, like matching timestamp and other qualities

In [ ]:
# Remove the latitudes and longitudes sections because those were only needed for comparison

county_annual_gdf = county_annual_gdf.drop(columns = ['lats', 'longs'], axis = 1)
state_annual_gdf = state_annual_gdf.drop(columns = ['lats', 'longs'], axis = 1)

## Part 6/6: Make County-Based, State-Based Population Estimates

In [76]:
# Make SLF abundance estimates given carrying capacity suggestions from the Lydemapr team's data, total land area, total forest area, and suggested
# reproduction rates

def est_pop(df: pd.DataFrame, dftype = 'county') -> list: # meant to accept take a dataframe with unique county-state-year or state-year data points
    def check_dftype(dftype):
        if dftype != 'county':
            dftype = 'state'
        return dftype

    dftype = check_dftype(dftype)
    
    abundances = []
    observed = df['slfs_obser'].to_list()
    observations = df['slf_observ'].to_list()
    observ_prev = df['y_slfs_obs'].to_list()
    land = df['land_area'].to_list()
    # forest = df['forest_area'].to_list() # still have to get this one
    # density = df['density'].to_list() # still have to get this one

    # Transform densities

    # density_transform = []
    # for i in range(0, len(density)):
    #     i str(density[i]) == 'Low':
    #         transformed = float(1) / float(3)
    #         density_transform = density_transform + [transformed]

    #     elif str(density[i]) == 'Medium':
    #         transformed = float(2) / float(3)
    #         density_transform = density_transform + [transformed]

    #     elif str(density[i]) == 'High':
    #         transformed = float(1)
    #         density_transform = density_transform + [transformed]

    #     else: # implies the current entry being evaluated has the value 'Unpopulated', meaning very little Lycorma delicatula are in the area
    #         transformed = 1e-5 # have to make it nonzero because when an area is unpopulated but a lanternfly is reported there is still some density
    #         density_transform = density_transform + [transformed]

    # Use multipliers to make the estimates based on the formulas:
    # E = Ey_1 + Q * A * (F + N) <--- uses formulas (A = l / q), for the estimated multiplier to represent land containing spotted lanternflies
    #     and (F = d * f / l), the ratio of all land that could be fully occupied by spotted lanternflies at "natural" carrying capacity
    #     and (N = (l - f) / l), the ratio of all non-forested area that could be occipied by spotted lanternflies at their own capacity
    # E = estimated population in number of individual SLFs
    # Ey_1 = last year's estimated population
    # Q = total quantity observed in the data point in number of individual SLFs for that year 
    # q = total number of observations made in the area, which includes observations of multiple individuals at a time;
    #     each observation is made in one unique square meter
    # f = square meterage of forested land in the area
    # l = square meterage of all land in the area
    # d = density multiplier based on the estimated percentage of whether or not the area has reached its carrying capacity for SLFs
    
    # Simplified, this formula is E = Ey_1 + Q * (d * f + (l - f)) / q
    
    for i in range(0, len(df)):
        if dftype == 'county':
            berks2014 = df.loc[df['year'] == 2014]
            berks2014 = berks2014.loc[berks2014['county'] == 'Berks']
            berks2014 = berks2014.loc[berks2014['state'] == 'Pennsylvania']
            berks2014 = berks2014.index.item()
        else:
            pa2014 = df.loc[df['year'] == 2014]
            pa2014 = pa2014.loc[pa2014['state'] == 'Pennsylvania']
            pa2014 = pa2014.index.item()
            
        Ey_1 = observ_prev[i]
        Q = observed[i]
        q = observations[i]
        # f = forest[i]
        l = land[i]
        # d = density_transformed[i]
        # E = Ey_1 + Q * (d * f + (l - f)) / q
        if (dftype == 'county' and i == berks2014) or (dftype == 'state' and i == pa2014):
            E = 30 # Estimate based on the number of eggs usually hatched in one egg mass, assuming one egg mass is responsible
        elif q == 0:
            E = Ey_1
        else:
            E = Ey_1 + Q * l / q
        abundances = abundances + [round(E)]
        # try:
        #     E = Q * 1e6 * l / q # this is the new estimate due to missing data
        #     print('Estimate:',E)
        #     abundances = abundances + [round(E)] # needs to be rounded because estimates can end up being floating-point values due to the
        #                                          # multiplication of fractions in the estimation process
        #     
        # except ZeroDivisionError:
        #     pass
        #     
        # else:
        #     E = 0
        #     print('Estimate:',E)
        #     abundances = abundances + [round(E)]
        #     
        # finally:
        #     pass
        
    return abundances

county_annual_data['slf_pop'] = est_pop(county_annual_data)
county_annual_data.head()

,county,state,land_area,water_area,year,slfs_obser,slf_observ,lats,longs,y_slfs_obs,y_slf_obse,geometry,slf_pop
0,Houston,Alabama,1501742250,4795418,2014,0,0,0,0,0,0,"POLYGON ((-85.712 31.197, -85.709 31.198, -85....",0
1,Choctaw,Alabama,2365900084,19114321,2014,0,0,0,0,0,0,"POLYGON ((-88.473 31.894, -88.469 31.93, -88.4...",0
2,Russell,Alabama,1660653961,15562947,2014,0,0,0,0,0,0,"POLYGON ((-85.435 32.318, -85.434 32.392, -85....",0
3,Sussex,Delaware,2424590442,674129051,2014,0,0,0,0,0,0,"POLYGON ((-75.723 38.83, -75.615 38.834, -75.5...",0
4,Jackson,Alabama,2792044612,126334711,2014,0,0,0,0,0,0,"MULTIPOLYGON (((-86.154 34.534, -86.15 34.534,...",0


In [77]:
len(county_annual_data.loc[county_annual_data['slf_pop'] != 0])

796

In [100]:
state_annual_data['slf_pop'] = est_pop(state_annual_data, dftype = 'state')
state_annual_data.head()

,state,land_area,water_area,year,slfs_obser,slf_observ,y_slfs_obs,y_slf_obse,geometry,slf_pop
0,New Mexico,314198519809,726531289,2014,0,0,0,0,"POLYGON ((-109.05 31.48, -109.05 31.5, -109.05...",0
1,South Dakota,196341670967,3387563375,2014,0,0,0,0,"POLYGON ((-104.06 44.998, -104.05 44.998, -104...",0
2,California,403673433805,20291632828,2014,0,0,0,0,"MULTIPOLYGON (((-118.6 33.479, -118.6 33.478, ...",0
3,Kentucky,102266755818,2384136185,2014,0,0,0,0,"MULTIPOLYGON (((-89.406 36.528, -89.399 36.542...",0
4,Alabama,131185561946,4581813708,2014,0,0,0,0,"MULTIPOLYGON (((-88.053 30.507, -88.051 30.509...",0


In [ ]:
len(state_annual_data.loc[state_annual_data['slf_pop'] != 0])

In [244]:
# Now aggregate the estimates from each county into the states

# def aggregate_pop(df: pd.DataFrame, gdf: gpd.GeoDataFrame) -> list:
#     aggregated_abundances = []
#     state_aggregates = {}
    
    # collect the populations of each county row per state
    
#     for i in range(0, len(df)):
#         df_state_year = str(df.at[i, 'state'] + ' ' + df.at[i, 'year'])
        
#         try:
#             state_aggregates[df_state_year] = state_aggregates[df_state_year] + int(df.at[i, 'slf_pop'])
        
#         except KeyError:
#             pass
        
#         else:
#             state_aggregates[df_state_year] = int(df.at[i, 'slf_pop'])
            
    # find the value of each associated state as you go through the state gdf
    
#     for i in range(0, len(gpd)):
#         gdf_state_year = str(gdf.at[i, 'state'] + ' ' + gdf.at[i, 'year'])
        
#         try:
#             pop = state_aggregates[gdf_state_year]
        
#         except KeyError:
#             print('No aggregate quantity found for State = ',gdf.at[i, 'state'],', Year =',gdf.at[i, 'year'])
        
#         else:
#             pop = 0

#         finally:
#             aggregated_abundances = aggregated_abundances + [pop]
    
#     return aggregated_abundances

# state_annual_gdf['slf_pop'] = aggregate_pop(gdf = county_annual_gdf)
# state_annual_gdf.head()

In [245]:
my_county_age_10 = county_annual_data.loc[county_annual_gdf['state'] == 'New Jersey']
my_county_age_10 = my_county_age_10.loc[county_annual_gdf['county'] == 'Ocean']
my_county_age_10 = my_county_age_10.loc[county_annual_gdf['year'] == 2015]
my_county_age_10.index.item()

5358

In [102]:
def yesteryear_pop(df: pd.DataFrame = county_annual_gdf, index_cutoff: int = 3108) -> list:
    previous_year = []
    for i in range(0, len(df)):
        if df.at[i, 'year'] == 2014:
            previous_year = previous_year + [0]
        else:
            previous_year = previous_year + [df.at[i - index_cutoff, 'slf_pop']]
    return previous_year

county_annual_data['slf_pop_ye'] = yesteryear_pop(df = county_annual_data, index_cutoff = 3108)
len(county_annual_data.loc[county_annual_data['slf_pop_ye'] != 0])

555

In [104]:
state_annual_data['slf_pop_ye'] = yesteryear_pop(df = state_annual_data, index_cutoff = 48)
# y = state_annual_data.loc[state_annual_data['state'] == 'Pennsylvania']
# y = y.loc[state_annual_data['slf_pop_ye'] != 0]
# y.head()

In [248]:
# Add a column for the previous years' abundances

# def yesteryear_pop(df: pd.DataFrame) -> list:
#     previous_year = []
#     for i in range(0, len(df)):
#         if df.at[i, 'year'] == 2014:
#             previous_year = previous_year + [0]
#         else:
#             this_state = str(df.at[i, 'state'])
#             this_year = int(df.at[i, 'year'])
#             state_condition = df.loc[df['state'] == this_state]
#             this_state_year = state_condition.loc[df['year'] == this_year]
            # try:
#             this_county = str(df.at[i, 'county'])
#             this_county_year = this_state_year.loc[this_state_year['county'] == this_county]
#             if len(this_county_year) == 0:
#                 print('No data points fit: County =',this_county,', State =',this_state,', Year =',this_year)
#             elif len(this_county_year) > 1:
#                 print('Multiple data points fit: County =',this_county,', State =',this_state,', Year =',this_year)
#             else:
#                 previous_county = df.loc[df['state'] == this_state]
#                 previous_county_state = previous_county.loc[df['county'] == this_county]
#                 previous_county_year = previous_county_state.loc[df['year'] == (this_year - 1)]
#                 if len(previous_county_year) == 0:
#                     print('No data points fit: County =',this_county,', State =',this_state,', Year =',(this_year - 1))
#                 elif len(previous_county_year) > 1:
#                     print('Multiple data points fit: County =',this_county,', State =',this_state,', Year =',(this_year - 1))
#                 else:
#                     index = previous_county_year.index.item()
#                     previous_year = previous_year + [int(df.at[index, 'slfs_obser'])]
            # except Exception:
            #     print('Couldn\'t find the county column')
            # else:
                # just do the other else block
            #     if len(this_state_year) == 0:
            #         print('No data points fit: State =',this_state,', Year=',this_year)
            #     elif len(this_state_year) > 1:
            #         print('Multiple data points fit: State =',this_state,', Year=',this_year)
            #     else:
            #         previous_state = df.loc[df['state'] == this_state]
            #         prevuous_state_year = previous_state.loc[df['year'] == (this_year - 1)]
            #         if len(previous_state_year) == 0:
            #             print('No data points fit: State =',this_state,', Year =',(this_year - 1))
            #         elif len(previous_county_year) > 1:
            #             print('Multiple data points fit: State =',this_state,', Year =',(this_year - 1))
            #         else:
            #             index = previous_state_year.index
            #             previous_year = previous_year + [int(df.at[index, 'slf_pop'])]
            #             print('Data point added')
            # finally:
            #     pass
#     return previous_year

# county_annual_data['slf_pop_yesteryear'] = yesteryear_pop(county_annual_data)
# county_annual_data.head()

In [106]:
def get_density(df: pd.DataFrame = county_annual_data, habitat: str = 'terrestrial') -> list:
    if habitat == 'terrestrial':
        densities = [float(df.at[0, 'slf_pop'] / df.at[0, 'land_area'])]
        for i in range(1, len(df)):
            densities = densities + [float(df.at[i, 'slf_pop'] * 1e6 / df.at[i, 'land_area'])] # 1e6 scalar makes density by square kilometer,
                                                                                                  # and prevents it from being by square meter
    elif habitat == 'aquatic': # may be reused for handling different species like aquatic predators
        densities = [float(df.at[0, 'slf_pop'] / df.at[0, 'water_area'])]
        for i in range(1, len(df)):
            densities = densities + [float(df.at[i, 'slf_pop'] * 1e6 / df.at[i, 'water_area'])]
    elif habitat == 'amphibious':
        densities = [float(df.at[0, 'slf_pop'] / df.at[0, 'land_area'] + df.at[0, 'water_area'])]
        for i in range(1, len(df)):
            densities = densities + [float(df.at[i, 'slf_pop'] * 1e6 / (df.at[i, 'land_area'] + df.at[i, 'water_area']))]
    else:
        print('Habitat unknown.')
    
    return densities

county_annual_data['slfdensity'] = get_density(df = county_annual_data, habitat = 'terrestrial')

In [107]:
county_annual_data.loc[county_annual_data['slfdensity'] != 0]

,county,state,land_area,water_area,year,geometry,slf_pop,slfdensity,slf_pop_ye
3098,Berks,Pennsylvania,2218077223,24220126,2014,"POLYGON ((-76.437 40.496, -76.434 40.496, -76....",30,1.352523e-02,0
6206,Berks,Pennsylvania,2218077223,24220126,2015,"POLYGON ((-76.437 40.496, -76.434 40.496, -76....",2218077223,1.000000e+06,30
6833,Montgomery,Pennsylvania,1250798248,11001813,2016,"POLYGON ((-75.696 40.242, -75.686 40.255, -75....",1250798248,1.000000e+06,0
8433,Lehigh,Pennsylvania,894414857,7450564,2016,"POLYGON ((-75.889 40.678, -75.855 40.693, -75....",894414857,1.000000e+06,0
9314,Berks,Pennsylvania,2218077223,24220126,2016,"POLYGON ((-76.437 40.496, -76.434 40.496, -76....",2218077225,1.000000e+06,2218077223
...,...,...,...,...,...,...,...,...,...
34067,Adams,Pennsylvania,1343104820,8147590,2024,"POLYGON ((-77.471 39.944, -77.421 39.981, -77....",1343104849,1.000000e+06,1343104832
34130,Fulton,Pennsylvania,1133252799,1305486,2024,"POLYGON ((-78.375 39.728, -78.371 39.732, -78....",1133252800,1.000000e+06,1133252799
34170,Portage,Ohio,1262380881,43125138,2024,"POLYGON ((-81.393 41.02, -81.393 41.029, -81.3...",1262380881,1.000000e+06,0
34176,Kent,Rhode Island,436588768,50686110,2024,"POLYGON ((-71.79 41.725, -71.72 41.726, -71.70...",436588768,1.000000e+06,0


In [110]:
state_annual_data['slfdensity'] = get_density(state_annual_data)
state_annual_data.head()

,state,land_area,water_area,year,slfs_obser,slf_observ,y_slfs_obs,y_slf_obse,geometry,slf_pop,slf_pop_ye,slfdensity
0,New Mexico,314198519809,726531289,2014,0,0,0,0,"POLYGON ((-109.05 31.48, -109.05 31.5, -109.05...",0,0,0.0
1,South Dakota,196341670967,3387563375,2014,0,0,0,0,"POLYGON ((-104.06 44.998, -104.05 44.998, -104...",0,0,0.0
2,California,403673433805,20291632828,2014,0,0,0,0,"MULTIPOLYGON (((-118.6 33.479, -118.6 33.478, ...",0,0,0.0
3,Kentucky,102266755818,2384136185,2014,0,0,0,0,"MULTIPOLYGON (((-89.406 36.528, -89.399 36.542...",0,0,0.0
4,Alabama,131185561946,4581813708,2014,0,0,0,0,"MULTIPOLYGON (((-88.053 30.507, -88.051 30.509...",0,0,0.0


In [112]:
# Get densities from the previous year

def get_yesteryear_density(df: pd.DataFrame = county_annual_data, index_cutoff: int = 3108):
    densities_previous_year = []
    for i in range(0, len(df)):
        if i < index_cutoff:
            densities_previous_year = densities_previous_year + [0]
        else:
            densities_previous_year = densities_previous_year + [df.at[i - index_cutoff, 'slfdensity']]
        
    return densities_previous_year

county_annual_data['y_slfdnsty'] = get_yesteryear_density(df = county_annual_data, index_cutoff = 3108)
county_annual_data.loc[county_annual_data['y_slfdnsty'] != 0].head()

,county,state,land_area,water_area,year,geometry,slf_pop,slfdensity,slf_pop_ye,y_slfdnsty
6206,Berks,Pennsylvania,2218077223,24220126,2015,"POLYGON ((-76.437 40.496, -76.434 40.496, -76....",2218077223,1.000000e+06,30,1.352523e-02
9314,Berks,Pennsylvania,2218077223,24220126,2016,"POLYGON ((-76.437 40.496, -76.434 40.496, -76....",2218077225,1.000000e+06,2218077223,1.000000e+06
9941,Montgomery,Pennsylvania,1250798248,11001813,2017,"POLYGON ((-75.696 40.242, -75.686 40.255, -75....",1250798249,1.000000e+06,1250798248,1.000000e+06
11541,Lehigh,Pennsylvania,894414857,7450564,2017,"POLYGON ((-75.889 40.678, -75.855 40.693, -75....",894414858,1.000000e+06,894414857,1.000000e+06
12422,Berks,Pennsylvania,2218077223,24220126,2017,"POLYGON ((-76.437 40.496, -76.434 40.496, -76....",2218077226,1.000000e+06,2218077225,1.000000e+06


In [113]:
state_annual_data['y_slfdnsty'] = get_yesteryear_density(df = state_annual_data, index_cutoff = 48)
state_annual_data.loc[state_annual_data['y_slfdnsty'] != 0].head()

,state,land_area,water_area,year,slfs_obser,slf_observ,y_slfs_obs,y_slf_obse,geometry,slf_pop,slf_pop_ye,slfdensity,y_slfdnsty
55,Pennsylvania,115881476238,3397613881,2015,2,2,0,0,"POLYGON ((-80.52 40.907, -80.52 40.911, -80.51...",115881476238,30,1.000000e+06,2.588852e-04
103,Pennsylvania,115881476238,3397613881,2016,5,5,2,2,"POLYGON ((-80.52 40.907, -80.52 40.911, -80.51...",115881476240,115881476238,1.000000e+06,1.000000e+06
151,Pennsylvania,115881476238,3397613881,2017,50,50,5,5,"POLYGON ((-80.52 40.907, -80.52 40.911, -80.51...",115881476243,115881476240,1.000000e+06,1.000000e+06
199,Pennsylvania,115881476238,3397613881,2018,311,311,50,50,"POLYGON ((-80.52 40.907, -80.52 40.911, -80.51...",115881476288,115881476243,1.000000e+06,1.000000e+06
247,Pennsylvania,115881476238,3397613881,2019,1153,1153,311,311,"POLYGON ((-80.52 40.907, -80.52 40.911, -80.51...",115881476549,115881476288,1.000000e+06,1.000000e+06


In [260]:
# test1 = slf_iNat_df.loc[slf_iNat_df['place_county_name'] == 'Ocean']
# test1 = test1.loc[slf_iNat_df['place_state_name'] == 'New Jersey']
# test1

In [261]:
# test = county_annual_data.loc[county_annual_data['county'] == 'Ocean']
# test = test.loc[county_annual_data['state'] == 'New Jersey']

In [262]:
# test

In [86]:
# Drop slf observation data that isn't about population abundance or density

county_annual_data = county_annual_data.drop(columns = ['lats', 'longs', 'slfs_obser', 'slf_observ', 'y_slfs_obs', 'y_slf_obse'], axis = 1)
state_annual_data = state_annual_data.drop(columns = ['lats', 'longs', 'slfs_obser', 'slf_observ', 'y_slfs_obs', 'y_slf_obse'], axis = 1)

# Confirm ideal dataframe

county_annual_data.head()

KeyError: "['lats', 'longs'] not found in axis"

In [116]:
county_annual_data.head()

,county,state,land_area,water_area,year,geometry,slf_pop,slfdensity,slf_pop_ye,y_slfdnsty
0,Houston,Alabama,1501742250,4795418,2014,"POLYGON ((-85.712 31.197, -85.709 31.198, -85....",0,0.0,0,0.0
1,Choctaw,Alabama,2365900084,19114321,2014,"POLYGON ((-88.473 31.894, -88.469 31.93, -88.4...",0,0.0,0,0.0
2,Russell,Alabama,1660653961,15562947,2014,"POLYGON ((-85.435 32.318, -85.434 32.392, -85....",0,0.0,0,0.0
3,Sussex,Delaware,2424590442,674129051,2014,"POLYGON ((-75.723 38.83, -75.615 38.834, -75.5...",0,0.0,0,0.0
4,Jackson,Alabama,2792044612,126334711,2014,"MULTIPOLYGON (((-86.154 34.534, -86.15 34.534,...",0,0.0,0,0.0


In [118]:
state_annual_data.head()

,state,land_area,water_area,year,slfs_obser,slf_observ,y_slfs_obs,y_slf_obse,geometry,slf_pop,slf_pop_ye,slfdensity,y_slfdnsty
0,New Mexico,314198519809,726531289,2014,0,0,0,0,"POLYGON ((-109.05 31.48, -109.05 31.5, -109.05...",0,0,0.0,0.0
1,South Dakota,196341670967,3387563375,2014,0,0,0,0,"POLYGON ((-104.06 44.998, -104.05 44.998, -104...",0,0,0.0,0.0
2,California,403673433805,20291632828,2014,0,0,0,0,"MULTIPOLYGON (((-118.6 33.479, -118.6 33.478, ...",0,0,0.0,0.0
3,Kentucky,102266755818,2384136185,2014,0,0,0,0,"MULTIPOLYGON (((-89.406 36.528, -89.399 36.542...",0,0,0.0,0.0
4,Alabama,131185561946,4581813708,2014,0,0,0,0,"MULTIPOLYGON (((-88.053 30.507, -88.051 30.509...",0,0,0.0,0.0


In [284]:
# Save shapefiles

county_annual_data.to_file('C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Modified Data Backups/county_annual_slf_est.shp')
state_annual_data.to_file('C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Modified Data Backups/state_annual_slf_est.shp')

# Save CSVs

county_annual_data.to_csv('C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Modified Data Backups/county_annual_slf_est.csv')
state_annual_data.to_csv('C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Modified Data Backups/state_annual_slf_est.csv')

# county_annual_data_csv = pd.DataFrame(county_annual_data)
# state_annual_data_csv = pd.DataFrame(state_annual_data)

# county_annual_data_csv.to_csv('C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Modified Data Backups/county_annual_slf_est.csv')
# state_annual_data_csv.to_csv('C:/Users/EK111/Documents/NJIT/Spotted Lanternfly Project URI Summer 2025/Modified Data Backups/state_annual_slf_est.csv')

## Extra: Map the Population Densities Each Year

In [122]:
maximum_density_c = max(list(county_annual_data['slfdensity']))
maximum_density_s = max(list(state_annual_data['slfdensity']))
print(maximum_density_c,',',maximum_density_s)

1445455.2447997208 , 1034678.1040833077


In [126]:
min_nonzero_density_c = min(list(county_annual_data['slfdensity'].loc[county_annual_data['slfdensity'] != 0]))
min_nonzero_density_s = min(list(state_annual_data['slfdensity'].loc[state_annual_data['slfdensity'] != 0]))
print(min_nonzero_density_c,',',min_nonzero_density_s)

0.00022482068410149993 , 2.4772499655824856e-06


In [ ]:
county_colors = []
for i in range(0, len(county_annual_data)):
    color = '#'
    if county_annual_data.at[i, 'slfdensity'] == 0:
        color = '#FFFFFF'
    else:
        red_intensity = 
        green_intensity = 
        blue_intensity = 
        red_hex = 
        green_hex = 
        blue_hex=
        color = color + red_hex + green_hex + blue_hex
    county_colors = county_colors + [color]

county_annual_data['color'] = county_colors

In [ ]:
state_colors = []
for i in range(0, len(state_annual_data)):
    color = '#'
    if state_annual_data.at[i, 'slfdensity'] == 0:
        color = '#FFFFFF'
    else:
        red_intensity = 
        green_intensity = 
        blue_intensity = 
        red_hex = 
        green_hex = 
        blue_hex=
        color = color + red_hex + green_hex + blue_hex
    state_colors = state_colors + [color]

state_annual_data['color'] = state_colors

In [ ]:
lightcolor = [255, 255, 255] # white
darkcolor = [128, 0, 0] # crimson
discrete_levels = 256
values = np.ones((N,4))
values[:, 0] = np.linspace(lightcolor[0]/256, darkcolor[0]/256, N)
values[:, 1] = np.linspace(lightcolor[1]/256, darkcolor[1]/256, N)
values[:, 2] = np.linspace(lightcolor[2]/256, darkcolor[2]/256, N)
colormap = ListedColormap(values)
fig, ax = plt.subplots(1, figsize=(16, 9))

In [ ]:
# 2014



In [ ]:
# 2015



In [ ]:
# 2016



In [ ]:
# 2017



In [ ]:
# 2018



In [ ]:
# 2019



In [ ]:
# 2020



In [ ]:
# 2021



In [ ]:
# 2022



In [ ]:
# 2023



In [ ]:
# 2024

